# Product Suggestion Agent

This agent handles customer inquiries about:
- Product recommendations
- Product comparisons
- Feature explanations
- Best matches for customer needs

## Setup

In [ ]:
# Import required libraries
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator

# Load configuration
from config_loader import load_config, get_model_name
config = load_config()
model_name = get_model_name(config)

## Define Agent State

In [ ]:
class AgentState(TypedDict):
    """State for product suggestion agent"""
    messages: Annotated[Sequence[BaseMessage], operator.add]

## Create Product Suggestion Agent

In [ ]:
# System prompt for product suggestion agent
PRODUCT_SUGGESTION_PROMPT = """You are a helpful sales assistant specializing in PRODUCT RECOMMENDATIONS and COMPARISONS.

Your responsibilities:
- Recommend products based on customer needs
- Compare different products objectively
- Explain product features and benefits
- Match products to customer requirements
- Provide honest assessments

Product categories (for demonstration):
- Electronics: Laptops, phones, tablets, headphones
- Home & Kitchen: Appliances, furniture, decor
- Clothing: Apparel, shoes, accessories
- Sports & Outdoors: Equipment, gear, fitness

When recommending:
- Ask clarifying questions about needs and budget
- Provide 2-3 options at different price points
- Explain pros/cons of each option
- Consider customer's stated preferences

Be enthusiastic but honest. Help customers make informed decisions.
"""

def create_product_suggestion_agent():
    """Create the product suggestion agent"""
    
    # Initialize LLM
    llm = ChatOpenAI(model=model_name, temperature=0.7)
    
    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", PRODUCT_SUGGESTION_PROMPT),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    # Create agent function
    def agent_node(state: AgentState):
        messages = state["messages"]
        response = llm.invoke(prompt.format_messages(messages=messages))
        return {"messages": [response]}
    
    # Create graph
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", agent_node)
    workflow.set_entry_point("agent")
    workflow.add_edge("agent", END)
    
    # Compile with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)
    
    return app

# Create the agent
product_suggestion_agent = create_product_suggestion_agent()
print("✅ Product Suggestion Agent created successfully!")

## Test the Agent

In [ ]:
# Test conversation
from uuid import uuid4

thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

# Test query 1
print("User: I need a laptop for gaming\n")
result = product_suggestion_agent.invoke(
    {"messages": [HumanMessage(content="I need a laptop for gaming")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)

In [ ]:
# Test query 2 (with context)
print("User: My budget is around $1500\n")
result = product_suggestion_agent.invoke(
    {"messages": [HumanMessage(content="My budget is around $1500")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)

In [ ]:
# Test query 3 (follow-up)
print("User: Which one has better battery life?\n")
result = product_suggestion_agent.invoke(
    {"messages": [HumanMessage(content="Which one has better battery life?")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)
print("\n✅ Agent maintained context across 3 turns!")